# Contador de fichas de dominó
**Inteligencia Artificial I — Proyecto 3**

Se sube una foto de fichas de dominó y el programa:
1. Cuenta cuántas fichas hay.
2. Lee el valor de cada ficha, por ejemplo `[3|5]`.
3. Cuenta las mulas (las dos mitades iguales).
4. Calcula la puntuación total (suma de todos los puntos).

In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
from google.colab import files

## 1. Subir la imagen

In [ ]:
subidas = files.upload()
nombre = list(subidas.keys())[0]
original = cv2.imread(nombre)

# REDIMENSIONAR A 1000 PX DE ANCHO PARA QUE TODAS LAS FOTOS SE PROCESEN IGUAL
escala = 1000 / original.shape[1]
img = cv2.resize(original, None, fx=escala, fy=escala)
cv2_imshow(img)

## 2. Escala de grises y filtro mediano
El filtro mediano quita la textura del fondo sin borrar los bordes de las fichas.

In [ ]:
gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gris = cv2.medianBlur(gris, 5)
cv2_imshow(gris)

## 3. Umbralización
- **Adaptativa:** compara cada píxel con su vecindario, funciona aunque la luz no sea pareja.
- **Global en 80:** deja solo lo oscuro, quita monedas o una goma.

Con `bitwise_and` nos quedamos con lo que cumple los dos umbrales.

In [ ]:
adaptativo = cv2.adaptiveThreshold(gris, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 601, 25)
ret, oscuro = cv2.threshold(gris, 80, 255, cv2.THRESH_BINARY_INV)
binaria = cv2.bitwise_and(adaptativo, oscuro)
cv2_imshow(binaria)

## 4. Rellenar los huecos de los puntos y apertura
Primero se rellenan los contornos y después se abre para quitar ruido. Si se abre antes, se pierden las esquinas de las fichas.

In [ ]:
# RELLENAR: DIBUJAR LOS CONTORNOS EXTERNOS RELLENOS
contornos, jerarquia = cv2.findContours(binaria, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
rellena = np.zeros_like(binaria)
cv2.drawContours(rellena, contornos, -1, 255, -1)

# APERTURA
kernel = np.ones((9, 9), np.uint8)
limpia = cv2.morphologyEx(rellena, cv2.MORPH_OPEN, kernel, iterations=2)
cv2_imshow(limpia)

## 5. Contornos de las fichas
Se ignoran las manchas muy chicas y las que tocan el borde de la foto (fichas cortadas).

In [ ]:
contornos, jerarquia = cv2.findContours(limpia, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
alto, ancho = limpia.shape
fichas = []
for c in contornos:
    x, y, w, h = cv2.boundingRect(c)
    toca_borde = x <= 1 or y <= 1 or x + w >= ancho - 1 or y + h >= alto - 1
    if cv2.contourArea(c) > 2000 and not toca_borde:
        fichas.append(c)

copia = img.copy()
cv2.drawContours(copia, fichas, -1, (0, 255, 0), 3)
cv2_imshow(copia)
print("Manchas encontradas:", len(fichas))

## 6. ¿Cuánto mide una ficha?
Para saber cuántas fichas hay en una mancha de fichas pegadas, necesitamos el área de **una** ficha.
Se toman las manchas que son claramente una ficha sola (sólidas y con forma de rectángulo 2:1) y se saca la mediana de sus áreas.
Si todas las fichas están pegadas, se usa el tamaño de los puntos: una ficha mide unas 110 veces el área de un punto.

In [ ]:
areas = []
for c in fichas:
    solidez = cv2.contourArea(c) / cv2.contourArea(cv2.convexHull(c))
    hull = cv2.convexHull(c)
    esquinas = cv2.approxPolyDP(hull, 0.04 * cv2.arcLength(hull, True), True)
    if solidez > 0.93 and len(esquinas) == 4:
        p = esquinas.reshape(4, 2)
        lados = [np.linalg.norm(p[i] - p[(i + 1) % 4]) for i in range(4)]
        if 1.7 < max(lados) / min(lados) < 2.7:
            areas.append(cv2.contourArea(c))

if areas:
    area_ficha = np.median(areas)
else:
    # SI TODAS ESTAN PEGADAS: UNA FICHA MIDE UNAS 110 VECES UN PUNTO
    ret, muy_blancos = cv2.threshold(gris, 140, 255, cv2.THRESH_BINARY)
    muy_blancos = cv2.bitwise_and(muy_blancos, cv2.erode(limpia, np.ones((7, 7), np.uint8)))
    muy_blancos = cv2.morphologyEx(muy_blancos, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    cp, j = cv2.findContours(muy_blancos, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    areas_puntos = [cv2.contourArea(p) for p in cp if cv2.contourArea(p) > 15]
    area_ficha = 110 * np.median(areas_puntos)

print("Área de una ficha:", int(area_ficha), "píxeles")

## 7. Puntos blancos
Umbral en 100: deja solo los puntos pintados, incluso los que tienen la pintura gastada. Se cuenta por color, no por forma, porque las posiciones vacías tienen círculos grabados.

In [ ]:
ret, blancos = cv2.threshold(gris, 100, 255, cv2.THRESH_BINARY)
cv2_imshow(blancos)
fondo = np.percentile(gris, 90)   # BRILLO DEL PAPEL

## 8. Contar las fichas
Para cada mancha, `n = redondeo(área de la mancha / área de una ficha)` dice cuántas fichas hay en ella.
Si son varias fichas pegadas, se revisa que la mancha sea **negra** (comparada con el brillo del papel) y que tenga puntos, para no contar un mouse o una tarjeta.

In [ ]:
# ORDENAR DE ARRIBA ABAJO Y DE IZQUIERDA A DERECHA (PARA NUMERARLAS)
fichas.sort(key=lambda c: (int((cv2.boundingRect(c)[1] + cv2.boundingRect(c)[3] / 2) // 80), cv2.boundingRect(c)[0]))

contadas = []   # (contorno, numero de fichas, centros de los puntos)
for c in fichas:
    n = round(cv2.contourArea(c) / area_ficha)
    if n == 0:
        continue
    # PUNTOS DENTRO DE LA MANCHA
    mascara = np.zeros_like(gris)
    cv2.drawContours(mascara, [cv2.convexHull(c)], -1, 255, -1)
    mascara = cv2.erode(mascara, np.ones((15, 15), np.uint8))
    puntos = cv2.bitwise_and(blancos, mascara)
    cp, j = cv2.findContours(puntos, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    centros = []
    for p in cp:
        x, y, w, h = cv2.boundingRect(p)
        if area_ficha * 0.004 < cv2.contourArea(p) < area_ficha * 0.05 and 0.5 < w / h < 2:
            centros.append((x + w / 2, y + h / 2))
    # SI SON VARIAS: QUE SEA NEGRA (NO MOUSE NI TARJETA) Y TENGA PUNTOS
    interior = np.zeros_like(gris)
    cv2.drawContours(interior, [c], -1, 255, -1)
    interior = cv2.erode(interior, np.ones((9, 9), np.uint8))
    negro = np.mean(gris[interior > 0] < 0.35 * fondo)
    if n >= 2 and (negro < 0.8 or len(centros) < n / 2):
        continue
    contadas.append((c, n, centros))

total = sum(n for c, n, centros in contadas)
copia = img.copy()
for c, n, centros in contadas:
    cv2.drawContours(copia, [c], -1, (0, 200, 0) if n == 1 else (255, 0, 255), 3)
cv2_imshow(copia)
print("Fichas contadas:", total)

## 9. Leer el valor de cada ficha
Con `approxPolyDP` se sacan las 4 esquinas de la ficha. La línea entre los puntos medios de los lados largos la divide en dos mitades,
y cada punto se asigna a la mitad que le toca con el signo del producto cruz.

In [ ]:
valores = []   # (contorno, esquinas, a, b, menor, mayor) de cada ficha suelta
for c, n, centros in contadas:
    if n != 1:
        continue
    x, y, w, h = cv2.boundingRect(c)
    # 4 ESQUINAS (SUBIR LA TOLERANCIA HASTA TENER 4)
    hull = cv2.convexHull(c)
    for eps in np.arange(0.01, 0.2, 0.01):
        esquinas = cv2.approxPolyDP(hull, eps * cv2.arcLength(hull, True), True).reshape(-1, 2)
        if len(esquinas) <= 4:
            break
    if len(esquinas) != 4:
        esquinas = np.array([[x, y], [x + w, y], [x + w, y + h], [x, y + h]])
    # LINEA QUE DIVIDE LA FICHA: PUNTOS MEDIOS DE LOS LADOS LARGOS
    if np.linalg.norm(esquinas[0] - esquinas[1]) > np.linalg.norm(esquinas[1] - esquinas[2]):
        a, b = (esquinas[0] + esquinas[1]) / 2, (esquinas[2] + esquinas[3]) / 2
    else:
        a, b = (esquinas[1] + esquinas[2]) / 2, (esquinas[3] + esquinas[0]) / 2
    # CADA PUNTO VA A LA MITAD QUE LE TOCA (SIGNO DEL PRODUCTO CRUZ)
    mitad1 = 0
    for (px, py) in centros:
        if (b[0] - a[0]) * (py - a[1]) - (b[1] - a[1]) * (px - a[0]) >= 0:
            mitad1 += 1
    menor, mayor = sorted([mitad1, len(centros) - mitad1])
    valores.append((c, esquinas, a, b, menor, mayor))

copia = img.copy()
for c, esquinas, a, b, menor, mayor in valores:
    cv2.line(copia, (int(a[0]), int(a[1])), (int(b[0]), int(b[1])), (255, 0, 0), 2)
for c, n, centros in contadas:
    for (px, py) in centros:
        cv2.circle(copia, (int(px), int(py)), 6, (0, 0, 255), 2)
cv2_imshow(copia)

## 10. Resultado
Cada ficha lleva un número y su valor: fichas en verde, mulas en naranja y fichas pegadas en morado.

In [ ]:
resultado = img.copy()
mulas = 0
puntuacion = 0
lista = []
numero = 0
for c, n, centros in contadas:
    numero += 1
    puntuacion += len(centros)
    x, y, w, h = cv2.boundingRect(c)
    for (px, py) in centros:
        cv2.circle(resultado, (int(px), int(py)), 6, (0, 0, 255), 2)
    if n == 1:
        ficha = [v for v in valores if v[0] is c][0]
        c, esquinas, a, b, menor, mayor = ficha
        if menor == mayor:
            mulas += 1
            color = (0, 200, 255)   # MULAS EN NARANJA
        else:
            color = (0, 200, 0)     # FICHAS EN VERDE
        texto = f"{numero} [{menor}|{mayor}]"
        cv2.drawContours(resultado, [esquinas.astype(np.int32)], -1, color, 3)
        cv2.line(resultado, (int(a[0]), int(a[1])), (int(b[0]), int(b[1])), (255, 0, 0), 2)
    else:
        color = (255, 0, 255)       # FICHAS PEGADAS EN MORADO
        texto = f"{numero}: {n} juntas"
        cv2.drawContours(resultado, [c], -1, color, 3)
    cv2.putText(resultado, texto, (x, max(y - 8, 24)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 5)
    cv2.putText(resultado, texto, (x, max(y - 8, 24)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
    lista.append(texto)

cv2_imshow(resultado)
print(f"Se detectaron {total} objetos en la imagen.")
print("Fichas:", "   ".join(lista))
print("Mulas:", mulas)
print("Puntuación total:", puntuacion)